eid data source: https://www.pxweb.bfs.admin.ch/pxweb/fr/px-x-1703030000_100/-/px-x-1703030000_100.px/table/tableViewLayout2/ $\newline$
2025 population data source: https://www.pxweb.bfs.admin.ch/pxweb/fr/px-x-0102020000_202/-/px-x-0102020000_202.px/
$\newline$
NDG data source: https://www.fedlex.admin.ch/eli/fga/2016/134/de
$\newline$
Note: this data analysis was done without any help from AI.


In [292]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import statsmodels.formula.api as smf

In [326]:
cantons_df = pd.read_csv("cantons_data.csv")
eid_df = pd.read_csv("eid_votation.csv")
ndg_df = pd.read_csv("NDG_data.csv")

In [327]:
cantons_df = cantons_df[cantons_df["Sex"] != "Sexe - total"]
cantons_df = cantons_df[cantons_df["Age"] != "100 ans ou plus"]
cantons_df = cantons_df[cantons_df["Canton"] != "Suisse"]
cantons_df = cantons_df[cantons_df["Canton"] != "Sans indication"]
col = ["Canton", "Age", "Sex", "Count", "Immigration", "Swiss citizenship acquisition"]
cantons_df = cantons_df[col]

cantons_df.head()

,Canton,Age,Sex,Count,Immigration,Swiss citizenship acquisition
332,Z�rich,18 ans,Homme,6227,29,106
333,Z�rich,19 ans,Homme,5993,34,74
334,Z�rich,20 ans,Homme,6065,39,63
335,Z�rich,21 ans,Homme,5925,32,42
336,Z�rich,22 ans,Homme,5819,38,28


In [328]:
cantons_df["Canton"].unique()

array(['Z�rich', 'Bern / Berne', 'Luzern', 'Uri', 'Schwyz', 'Obwalden',
       'Nidwalden', 'Glarus', 'Zug', 'Fribourg / Freiburg', 'Solothurn',
       'Basel-Stadt', 'Basel-Landschaft', 'Schaffhausen',
       'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'St. Gallen',
       'Graub�nden / Grigioni / Grischun', 'Aargau', 'Thurgau', 'Ticino',
       'Vaud', 'Valais / Wallis', 'Neuch�tel', 'Gen�ve', 'Jura'],
      dtype=object)

In [329]:
def fix_cantons(cantons_df):
    cantons_df = cantons_df.replace('Z�rich','Zurich')
    cantons_df = cantons_df.replace('Bern / Berne','Berne')
    cantons_df = cantons_df.replace('Fribourg / Freiburg','Fribourg')
    cantons_df = cantons_df.replace('Graub�nden / Grigioni / Grischun','Grischun')
    cantons_df = cantons_df.replace('Valais / Wallis','Valais')
    cantons_df = cantons_df.replace('Neuch�tel','Neuchatel')
    cantons_df = cantons_df.replace('Gen�ve','Geneve')

    return cantons_df

cantons_df = fix_cantons(cantons_df)
eid_df = fix_cantons(eid_df)



In [330]:
cantons_df["Canton"].unique()

array(['Zurich', 'Berne', 'Luzern', 'Uri', 'Schwyz', 'Obwalden',
       'Nidwalden', 'Glarus', 'Zug', 'Fribourg', 'Solothurn',
       'Basel-Stadt', 'Basel-Landschaft', 'Schaffhausen',
       'Appenzell Ausserrhoden', 'Appenzell Innerrhoden', 'St. Gallen',
       'Grischun', 'Aargau', 'Thurgau', 'Ticino', 'Vaud', 'Valais',
       'Neuchatel', 'Geneve', 'Jura'], dtype=object)

In [331]:
cantons_population_df = cantons_df[["Canton", "Count"]].groupby(["Canton"]).sum()
cantons_population_df.head()

,Count
Canton,
Aargau,439719
Appenzell Ausserrhoden,37926
Appenzell Innerrhoden,11981
Basel-Landschaft,189200
Basel-Stadt,104868


In [332]:
eid_df.head()

,Canton,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,yes_percentage
0,Suisse,5641040,2796897,49.58,2747948,1384586,1363362,50.39
1,Zurich,976490,509297,52.16,503923,274702,229221,54.51
2,Berne,750745,358855,47.80,354158,171771,182387,48.50
3,Luzern,288363,153604,53.27,151603,80261,71342,52.94
4,Uri,27226,12731,46.76,12567,5109,7458,40.65


In [333]:
cantons_df.head()

,Canton,Age,Sex,Count,Immigration,Swiss citizenship acquisition
332,Zurich,18 ans,Homme,6227,29,106
333,Zurich,19 ans,Homme,5993,34,74
334,Zurich,20 ans,Homme,6065,39,63
335,Zurich,21 ans,Homme,5925,32,42
336,Zurich,22 ans,Homme,5819,38,28


In [334]:
# for later graphs that have to do with age groups

age_groups = [range(18, 24), range(24,30), range(30, 36), range(36, 42), range(42, 48), range(48, 54), range(54, 60), range(60, 66),
              range(66, 72), range(72, 78), range(78, 84), range(84, 90), range(90, 96)]
def age_sort(df, age_groups):
    for i in range(95, 100):
        df = df[df["Age"] != str(i) + " ans"]
    # group by ages to count population groups
    for age_group in age_groups:
        new_string = str(age_group[0]) + " - " + str(age_group[-1])
        for age in age_group:
            age_string = str(age) + " ans"
            df = df.replace(age_string, new_string)
    df.reset_index()
    return df
        

In [335]:
def age_transform(df):
    for i in range(18, 100):
        age_string = str(i) + " ans"
        df = df.replace(age_string, i)
    return df

In [336]:
# represent demographics with percentages instead of count
def canton_proportions(df):
    df["Count"] = df["Count"] / df.groupby("Canton")["Count"].transform('sum')
    return df

In [337]:
cantons_df = canton_proportions(cantons_df)
cantons_df = cantons_df.rename(columns={"Count":"Proportion"})
cantons_df = cantons_df.replace("Homme","Male")
cantons_df = cantons_df.replace("Femme","Female")
cantons_df.head()

,Canton,Age,Sex,Proportion,Immigration,Swiss citizenship acquisition
332,Zurich,18 ans,Male,0.006559,29,106
333,Zurich,19 ans,Male,0.006313,34,74
334,Zurich,20 ans,Male,0.006389,39,63
335,Zurich,21 ans,Male,0.006241,32,42
336,Zurich,22 ans,Male,0.006129,38,28


In [338]:
#cantons_df = age_sort(cantons_df, age_groups)
cantons_df = age_transform(cantons_df)
cantons_df = cantons_df.groupby(["Canton", "Age", "Sex"]).sum()
cantons_df.head(10)

/var/folders/22/_dymvqsd4tgfb0zzj_kpnj340000gn/T/ipykernel_2063/749895442.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(age_string, i)


Proportion  Immigration  Swiss citizenship acquisition
Canton Age Sex                                                           
Aargau 18  Female    0.005881            4                             27
           Male      0.006156           11                             33
       19  Female    0.005886            9                             26
           Male      0.006318           11                             30
       20  Female    0.005799           10                             19
           Male      0.006334            2                             18
       21  Female    0.005981            8                             16
           Male      0.006297           12                             18
       22  Female    0.005831           10                             22
           Male      0.006274            7                             15

In [339]:
cantons_df = cantons_df.reset_index()
#cantons_df.pivot(columns=["Age", "Sex"], index="Canton", values=["Proportion"])
cantons_df.head()

,Canton,Age,Sex,Proportion,Immigration,Swiss citizenship acquisition
0,Aargau,18,Female,0.005881,4,27
1,Aargau,18,Male,0.006156,11,33
2,Aargau,19,Female,0.005886,9,26
3,Aargau,19,Male,0.006318,11,30
4,Aargau,20,Female,0.005799,10,19


In [340]:
cantons_df["Age_avg"] = cantons_df["Proportion"] * cantons_df["Age"]
ols_cantons = pd.DataFrame()
ols_cantons["Age_avg"] = cantons_df.groupby(["Canton"])["Age_avg"].sum()
ols_cantons["Immigration"] = cantons_df.groupby(["Canton"])["Immigration"].sum()
ols_cantons["swiss_citizenship_acquisition"] = cantons_df.groupby(["Canton"])["Swiss citizenship acquisition"].sum()

In [341]:
eid_df = eid_df.groupby(["Canton"]).sum()
eid_df.head(10)

,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,yes_percentage
Canton,,,,,,,
Aargau,447798,232292,51.87,230470,113121,117349,49.08
Appenzell Ausserrhoden,39526,21896,55.40,21652,9949,11703,45.95
Appenzell Innerrhoden,12385,6006,48.49,5859,2644,3215,45.13
Basel-Landschaft,192244,96303,50.09,93940,46245,47695,49.23
Basel-Stadt,114378,55409,48.44,54544,30976,23568,56.79
Berne,750745,358855,47.80,354158,171771,182387,48.50
Fribourg,219163,100846,46.01,98853,49792,49061,50.37
Geneve,285837,119086,41.66,112906,62308,50598,55.19
Glarus,26836,12159,45.31,12008,4965,7043,41.35


In [342]:
data_df = pd.merge(left=eid_df, right=ols_cantons, left_on="Canton", right_on="Canton", how="inner")
data_df

,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,yes_percentage,Age_avg,Immigration,swiss_citizenship_acquisition
Canton,,,,,,,,,,
Aargau,447798,232292,51.87,230470,113121,117349,49.08,52.583427,925,2157
Appenzell Ausserrhoden,39526,21896,55.40,21652,9949,11703,45.95,53.506565,114,93
Appenzell Innerrhoden,12385,6006,48.49,5859,2644,3215,45.13,52.388031,17,21
Basel-Landschaft,192244,96303,50.09,93940,46245,47695,49.23,54.861559,520,947
Basel-Stadt,114378,55409,48.44,54544,30976,23568,56.79,53.480070,555,1023
Berne,750745,358855,47.80,354158,171771,182387,48.50,53.538567,1817,2366
Fribourg,219163,100846,46.01,98853,49792,49061,50.37,51.279168,528,565
Geneve,285837,119086,41.66,112906,62308,50598,55.19,51.223613,2465,2132
Glarus,26836,12159,45.31,12008,4965,7043,41.35,53.957518,64,78


In [343]:
a1 = sorted(ndg_df["Canton"].unique().tolist())
a2 = sorted(ndg_df["Canton"].unique().tolist())
a1 == a2

True

In [344]:
data_df = data_df.reset_index()

In [345]:
cantons_df["Canton"].unique()

array(['Aargau', 'Appenzell Ausserrhoden', 'Appenzell Innerrhoden',
       'Basel-Landschaft', 'Basel-Stadt', 'Berne', 'Fribourg', 'Geneve',
       'Glarus', 'Grischun', 'Jura', 'Luzern', 'Neuchatel', 'Nidwalden',
       'Obwalden', 'Schaffhausen', 'Schwyz', 'Solothurn', 'St. Gallen',
       'Thurgau', 'Ticino', 'Uri', 'Valais', 'Vaud', 'Zug', 'Zurich'],
      dtype=object)

In [346]:
# add language, 0 is French, 1 German, 2 French and German, 3 Italian
cantons_lang = {
    "Aargau": "German",
    "Appenzell Ausserrhoden": "German",
    "Appenzell Innerrhoden": "German",
    "Basel-Landschaft": "German",
    "Basel-Stadt": "German",
    "Berne": "Bilingual",
    "Fribourg": "Bilingual", 
    "Geneve": "French",
    "Glarus": "German",
    "Grischun": "Bilingual",
    "Jura": "French",
    "Luzern": "German",
    "Neuchatel": "French",
    "Nidwalden": "German",
    "Obwalden": "German",
    "Schaffhausen": "German",
    "Schwyz": "German",
    "Solothurn": "German",
    "St. Gallen": "German",
    "Thurgau": "German",
    "Ticino": "Italian",
    "Uri": "German",
    "Valais": "Bilingual",
    "Vaud": "French",
    "Zug": "German",
    "Zurich": "German"
}
data_df["Language"] = data_df["Canton"].map(lambda x: cantons_lang[x])

In [347]:
#https://en.wikipedia.org/wiki/List_of_Swiss_cantons_by_GRDP
cantons_gdp = {
    "Aargau": 67224,
    "Appenzell Ausserrhoden": 67341,
    "Appenzell Innerrhoden": 75526,
    "Basel-Landschaft": 77693,
    "Basel-Stadt": 209782,
    "Berne": 85151,
    "Fribourg": 64502, 
    "Geneve": 119644,
    "Glarus": 75430,
    "Grischun": 82817,
    "Jura": 78546,
    "Luzern": 75544,
    "Neuchatel": 106165,
    "Nidwalden": 74952,
    "Obwalden": 74902,
    "Schaffhausen": 100959,
    "Schwyz": 70739,
    "Solothurn": 73803,
    "St. Gallen": 85320,
    "Thurgau": 70504,
    "Ticino": 102190,
    "Uri": 58392,
    "Valais": 61387,
    "Vaud": 78021,
    "Zug": 192958,
    "Zurich": 104620
}
data_df["gdp_per_capita"] = data_df["Canton"].map(lambda x: cantons_gdp[x])

In [348]:
#https://www.geo-ref.net/ph/che.htm
density = {
    "Aargau": 528,
    "Appenzell Ausserrhoden": 234,
    "Appenzell Innerrhoden": 96,
    "Basel-Landschaft": 581,
    "Basel-Stadt": 5441,
    "Berne": 183,
    "Fribourg": 217, 
    "Geneve": 1883,
    "Glarus": 62,
    "Grischun": 29,
    "Jura": 89,
    "Luzern": 306,
    "Neuchatel": 223,
    "Nidwalden": 164,
    "Obwalden": 81,
    "Schaffhausen": 296,
    "Schwyz": 198,
    "Solothurn": 366,
    "St. Gallen": 276,
    "Thurgau": 131,
    "Ticino": 347,
    "Uri": 35,
    "Valais": 303,
    "Vaud": 71,
    "Zug": 559,
    "Zurich": 974
}
data_df["population_density"] = data_df["Canton"].map(lambda x: cantons_gdp[x])

In [349]:
data_df.sort_values(by=["yes_percentage"]).head(50)

,Canton,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,yes_percentage,Age_avg,Immigration,swiss_citizenship_acquisition,Language,gdp_per_capita,population_density
21,Uri,27226,12731,46.76,12567,5109,7458,40.65,53.246437,43,43,German,58392,58392
16,Schwyz,109052,62804,57.59,62475,25512,36963,40.84,52.883302,250,407,German,70739,70739
8,Glarus,26836,12159,45.31,12008,4965,7043,41.35,53.957518,64,78,German,75430,75430
14,Obwalden,27787,15161,54.56,14763,6286,8477,42.58,53.108310,69,40,German,74902,74902
19,Thurgau,182141,96916,53.21,94526,40920,53606,43.29,52.801594,461,753,German,70504,70504
10,Jura,54885,23225,42.32,22710,9889,12821,43.54,53.078697,164,135,French,78546,78546
15,Schaffhausen,54628,37260,68.21,35447,15547,19900,43.86,54.462871,155,322,German,100959,100959
22,Valais,235721,109326,46.38,106552,47148,59404,44.25,53.342509,819,839,Bilingual,61387,61387
2,Appenzell Innerrhoden,12385,6006,48.49,5859,2644,3215,45.13,52.388031,17,21,German,75526,75526
1,Appenzell Ausserrhoden,39526,21896,55.40,21652,9949,11703,45.95,53.506565,114,93,German,67341,67341


In [350]:
model = smf.ols(formula='yes_percentage ~ Age_avg + C(Language) + Immigration + gdp_per_capita + population_density', data=data_df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         yes_percentage   R-squared:                       0.705
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     7.574
Date:                Mon, 04 May 2026   Prob (F-statistic):           0.000297
Time:                        14:50:47   Log-Likelihood:                -61.654
No. Observations:                  26   AIC:                             137.3
Df Residuals:                      19   BIC:                             146.1
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 88

t statistic over 3 is considered significant. 


In [351]:
data_df.head()

,Canton,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,yes_percentage,Age_avg,Immigration,swiss_citizenship_acquisition,Language,gdp_per_capita,population_density
0,Aargau,447798,232292,51.87,230470,113121,117349,49.08,52.583427,925,2157,German,67224,67224
1,Appenzell Ausserrhoden,39526,21896,55.40,21652,9949,11703,45.95,53.506565,114,93,German,67341,67341
2,Appenzell Innerrhoden,12385,6006,48.49,5859,2644,3215,45.13,52.388031,17,21,German,75526,75526
3,Basel-Landschaft,192244,96303,50.09,93940,46245,47695,49.23,54.861559,520,947,German,77693,77693
4,Basel-Stadt,114378,55409,48.44,54544,30976,23568,56.79,53.480070,555,1023,German,209782,209782


In [352]:
cantons_df_ = data_df[["Canton", "Age_avg", "Immigration", "Language", "swiss_citizenship_acquisition", "Language", "gdp_per_capita", "population_density"]]
ndg_df = pd.merge(left=ndg_df, right=cantons_population_df, left_on="Canton", right_on="Canton", how="inner")

In [353]:
data_df.head()

,Canton,Voter registrations,Vote count,Participation %,Valid votes,Yes,No,yes_percentage,Age_avg,Immigration,swiss_citizenship_acquisition,Language,gdp_per_capita,population_density
0,Aargau,447798,232292,51.87,230470,113121,117349,49.08,52.583427,925,2157,German,67224,67224
1,Appenzell Ausserrhoden,39526,21896,55.40,21652,9949,11703,45.95,53.506565,114,93,German,67341,67341
2,Appenzell Innerrhoden,12385,6006,48.49,5859,2644,3215,45.13,52.388031,17,21,German,75526,75526
3,Basel-Landschaft,192244,96303,50.09,93940,46245,47695,49.23,54.861559,520,947,German,77693,77693
4,Basel-Stadt,114378,55409,48.44,54544,30976,23568,56.79,53.480070,555,1023,German,209782,209782


In [354]:
ndg_df = pd.merge(left=ndg_df, right=cantons_df_, left_on="Canton", right_on="Canton", how="inner")

In [355]:
ndg_df["Signatures"] = (ndg_df["Signatures"] * 10**3)/ndg_df["Count"]

In [356]:
ndg_df.head()

,Canton,Signatures,Count,Age_avg,Immigration,Language,swiss_citizenship_acquisition,Language,gdp_per_capita,population_density
0,Zurich,14.630041,949348,51.748447,3387,German,7585,German,104620,104620
1,Berne,15.313103,733816,53.538567,1817,Bilingual,2366,Bilingual,85151,85151
2,Luzern,7.924254,284317,51.912235,575,German,982,German,75544,75544
3,Uri,7.106637,27017,53.246437,43,German,43,German,58392,58392
4,Schwyz,4.314232,107551,52.883302,250,German,407,German,70739,70739


In [357]:
model = smf.ols(formula='Signatures ~ Age_avg + Immigration + Count + swiss_citizenship_acquisition + gdp_per_capita + population_density', data=ndg_df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             Signatures   R-squared:                       0.499
Model:                            OLS   Adj. R-squared:                  0.374
Method:                 Least Squares   F-statistic:                     3.992
Date:                Mon, 04 May 2026   Prob (F-statistic):             0.0113
Time:                        14:50:53   Log-Likelihood:                -78.328
No. Observations:                  26   AIC:                             168.7
Df Residuals:                      20   BIC:                             176.2
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     